# Clonamos el repositorio con los modelos y herramientas¶

In [1]:
!git clone https://github.com/dannasalazar11/Msc_thesis.git

Cloning into 'Msc_thesis'...
remote: Enumerating objects: 554, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 554 (delta 62), reused 0 (delta 0), pack-reused 427 (from 1)
Receiving objects: 100% (554/554), 52.73 MiB | 39.44 MiB/s, done.
Resolving deltas: 100% (339/339), done.


In [2]:
import sys
sys.path.append('/kaggle/working/Msc_thesis')

from TGARNet.utils import get_segmented_data
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')


import tensorflow as tf
import numpy as np
import random
import os

# Establecer semilla
seed = 42

# Semillas para módulos principales
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

2025-12-01 16:24:46.425853: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764606286.686965      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764606286.759193      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Importar base de datos segmentada (Segmentos de 4 seg con translape del 50%, es decir, de 2 seg)

In [3]:
from gmrrnet_adhd.models.spatio_temporal import prepare_streams_4s

X, y, sbjs = get_segmented_data()
X.shape, y.shape, len(sbjs)

((8213, 19, 512), (8213, 2), 8213)

## Preprocesamiento de los datos mencionado por la propuesta

| Variable   | Forma resultante | Cálculo exacto                                                                                                                                                                                                                |
| ---------- | ---------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **`freq`** | `(N, 20, 1)`     | - PSD con `welch(signal, fs=128, nperseg=512)`.<br>- Potencia media en **20 bandas log‑espaciadas** entre 1 Hz y 64 Hz.<br>- Promedio sobre canales → vector de 20.<br>- Se añade un eje final de tamaño 1.                   |
| **`temp`** | `(N, 10, 1)`     | - Se recortan 510 muestras (de 512).<br>- Se dividen en **10 ventanas** consecutivas de 51 muestras (≈ 400 ms).<br>- **Media aritmética** dentro de cada ventana promediando canales.<br>- Se añade un eje final de tamaño 1. |
| **`spat`** | `(N, C, 1)`      | - Para cada canal: **RMS** del segmento `sqrt(mean(x**2))`.<br>- Se añade un eje final de tamaño 1.                                                                                                                           |

In [4]:
freq, temp, spat = prepare_streams_4s(X, fs=128)

freq.shape, temp.shape, spat.shape

((8213, 20, 1), (8213, 10, 1), (8213, 19, 1))

In [5]:
from sklearn.preprocessing import StandardScaler

scaler_f = StandardScaler().fit(freq.reshape(-1, 20))
scaler_t = StandardScaler().fit(temp.reshape(-1, 10))
scaler_s = StandardScaler().fit(spat.reshape(-1, spat.shape[1]))

freq = scaler_f.transform(freq.reshape(-1, 20)).reshape(freq.shape)
temp = scaler_t.transform(temp.reshape(-1, 10)).reshape(temp.shape)
spat = scaler_s.transform(spat.reshape(-1, spat.shape[1])).reshape(spat.shape)

# Importamos el modelo y definimos los hiperparámetros

In [6]:
from TGARNet.models.multi_stream import build_eeg_attention_model
from tensorflow.keras.optimizers import Adam

model_name="spatio_temporal"

model_args =    {'freq_shape' : freq.shape[1:],   # (20,1)
                 'temp_shape' : temp.shape[1:],   # (10,1)
                 'spat_shape' : spat.shape[1:]}   # (19,1)

compile_args = {'optimizer':lambda: Adam(1e-4, clipnorm=1.0),
    "loss": "categorical_crossentropy",
    "metrics": ["accuracy"]
}

model = build_eeg_attention_model(
    **model_args
)

model.summary()

I0000 00:00:1764606392.262619      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1764606392.263189      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "EEG_Attention_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ freq_input (InputLayer)   │ (None, 20, 1)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ temp_input (InputLayer)   │ (None, 10, 1)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ spat_input (InputLayer)   │ (None, 19, 1)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast (Cast)               │ (None, 20, 1)          │              0 │ freq_input[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_2 (Cast)             │ (None, 10, 1)          │              0 │ temp_input[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_4 (Cast)             │ (None, 19, 1)          │              0 │ spat_input[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 20, 64)         │            128 │ cast[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_6 (Dense)           │ (None, 10, 64)         │            128 │ cast_2[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_12 (Dense)          │ (None, 19, 64)         │            128 │ cast_4[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ positional_encoding       │ (None, 20, 64)         │              0 │ dense[0][0]            │
│ (PositionalEncoding)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ positional_encoding_1     │ (None, 10, 64)         │              0 │ dense_6[0][0]          │
│ (PositionalEncoding)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ positional_encoding_2     │ (None, 19, 64)         │              0 │ dense_12[0][0]         │
│ (PositionalEncoding)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_1 (Cast)             │ (None, 20, 64)         │              0 │ positional_encoding[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_3 (Cast)             │ (None, 10, 64)         │              0 │ positional_encoding_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ cast_5 (Cast)             │ (None, 19, 64)         │              0 │ positional_encoding_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ transformer_encoder_block │ (None, 20, 64)         │         83,200 │ cast_1[0][0]           │
│ (TransformerEncoderBlock) │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ transformer_encoder_bloc… │ (None, 10, 64)         │         83,200 │ cast_3[0][0]           │
│ (TransformerEncoderBlock) │                        │                │                        │
├──────────────────────

 Total params: 574,082 (2.19 MB)

 Trainable params: 574,082 (2.19 MB)

 Non-trainable params: 0 (0.00 B)

# Resultados - Leave 24 Subjects Out

In [7]:
import os

import pickle

with open("/kaggle/input/ieee-tdah-control-database/folds.pkl", "rb") as f:
    folds = pickle.load(f)

In [8]:
import numpy as np
import random
from collections import defaultdict
from copy import deepcopy

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    cohen_kappa_score,
    roc_auc_score
)


def SGKF(
    model_builder, X, y, sbjs, model_args, compile_args, folds,
    model_name='', delta=10, seed=42
):
    all_fold_metrics = []
    models = {}

    # ---------------------------------------------------------
    # 1. Construir un diccionario sujeto → clase (0 CTRL, 1 ADHD)
    # ---------------------------------------------------------
    y_classes = np.argmax(y, axis=1)
    subject_label = {
        sbj: y_classes[sbjs.index(sbj)] for sbj in set(sbjs)
    }

    # Extraer las tres vistas
    freq, temp, spat = X

    # ===============================
    # INICIO VALIDACIÓN CRUZADA
    # ===============================
    for fold, (train_subjects, test_subjects) in enumerate(folds):
        print(f"\n{'-'*60}")
        print(f"Fold {fold+1}/{len(folds)}  |  Test subjects: {test_subjects}")
        print(f"{'-'*60}")

        # Índices de entrenamiento y test
        train_idx = [i for i, s in enumerate(sbjs) if s in train_subjects]
        test_idx  = [i for i, s in enumerate(sbjs) if s in test_subjects]

        # --------------------------------------------
        # 2. SELECCIÓN ESTRATIFICADA DE VALIDACIÓN
        # --------------------------------------------
        train_ADHD = [s for s in train_subjects if subject_label[s] == 1]
        train_CTRL = [s for s in train_subjects if subject_label[s] == 0]

        rng = np.random.default_rng(seed + fold)
        val_ADHD = rng.choice(train_ADHD, size=8, replace=False)
        val_CTRL = rng.choice(train_CTRL, size=8, replace=False)

        val_subjects = set(val_ADHD.tolist() + val_CTRL.tolist())
        val_idx = [i for i, s in enumerate(sbjs) if s in val_subjects]

        train_idx_final = [i for i in train_idx if sbjs[i] not in val_subjects]

        # -----------------------------
        # 3. DATOS FINALES (CORREGIDO)
        # -----------------------------
        X_train_final = [
            freq[train_idx_final],
            temp[train_idx_final],
            spat[train_idx_final]
        ]

        X_val = [
            freq[val_idx],
            temp[val_idx],
            spat[val_idx]
        ]

        X_test = [
            freq[test_idx],
            temp[test_idx],
            spat[test_idx]
        ]

        y_train_final = y[train_idx_final]
        y_val         = y[val_idx]
        y_test        = y[test_idx]

        sbjs_test = [sbjs[i] for i in test_idx]

        # -----------------------------
        # 4. MODELO Y ENTRENAMIENTO
        # -----------------------------
        tf.keras.backend.clear_session()

        # Reproducibilidad
        np.random.seed(seed + fold)
        random.seed(seed + fold)
        tf.random.set_seed(seed + fold)

        # Callbacks
        early_stopping = EarlyStopping(
            monitor='val_loss', patience=30, min_delta=1e-4,
            restore_best_weights=True, verbose=1
        )
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=30,
            min_lr=1e-6, verbose=1
        )

        # Construcción y compilación del modelo
        model = model_builder(**model_args)
        compile_args_local = deepcopy(compile_args)

        if callable(compile_args_local["optimizer"]):
            compile_args_local["optimizer"] = compile_args_local["optimizer"]()

        model.compile(**compile_args_local)

        # Entrenamiento
        model.fit(
            X_train_final, y_train_final,
            epochs=100,
            validation_data=(X_val, y_val),
            verbose=0,
            batch_size=16,
            # callbacks=[early_stopping, reduce_lr]
        )

        # -----------------------------
        # 5. PREDICCIÓN Y EVALUACIÓN
        # -----------------------------
        y_pred_probs = model.predict(X_test)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.argmax(y_test, axis=1)

        fold_metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'kappa': cohen_kappa_score(y_true, y_pred),
            'auc': roc_auc_score(y_true, y_pred_probs[:, 1])
        }

        print(f"\nFold {fold+1} Metrics: {fold_metrics}")

        all_fold_metrics.append(fold_metrics)
        models[fold] = model

        # Accuracy por sujeto
        subject_correct = defaultdict(list)
        for yt, yp, sbj in zip(y_true, y_pred, sbjs_test):
            subject_correct[sbj].append(int(yt == yp))

        print("Average accuracy per test subject:")
        for sbj in test_subjects:
            acc_sbj = np.mean(subject_correct.get(sbj, []))
            print(f"  {sbj}: {acc_sbj:.4f}")

    # ---------------------------------------------------
    # 6. REPORTE FINAL
    # ---------------------------------------------------
    print("\n" + "="*50)
    print("Cross-Validation Final Results")
    print("="*50)

    mean_metrics = {}
    for key in all_fold_metrics[0].keys():
        values = [f[key] for f in all_fold_metrics]
        mean_metrics[f'mean_{key}'] = np.mean(values)
        mean_metrics[f'std_{key}'] = np.std(values)

    print("Individual Fold Accuracies:")
    for i, f in enumerate(all_fold_metrics):
        print(f"  Fold {i+1}: {f['accuracy']:.4f}")

    print("\nAverage Performance across all folds:")
    for key, value in mean_metrics.items():
        print(f"  {key}: {value:.4f}")

    return all_fold_metrics

In [9]:
import numpy as np

X_total = [freq, temp, spat]            # each array shape (N, …)

results = {}

for i in range(5):
    result = SGKF(build_eeg_attention_model, X_total, y, sbjs, model_args, compile_args, folds, model_name='multi_stream')
    results[i] = result


------------------------------------------------------------
Fold 1/5  |  Test subjects: ['v28p', 'v274', 'v1p', 'v231', 'v22p', 'v29p', 'v206', 'v238', 'v31p', 'v35p', 'v177', 'v200', 'v112', 'v113', 'v48p', 'v140', 'v131', 'v125', 'v55p', 'v143', 'v43p', 'v305', 'v134', 'v114']
------------------------------------------------------------


I0000 00:00:1764606425.756859      74 service.cc:148] XLA service 0x78dd34014980 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764606425.758047      74 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1764606425.758067      74 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1764606429.435203      74 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1764606443.137039      74 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


46/46 ━━━━━━━━━━━━━━━━━━━━ 7s 84ms/step

Fold 1 Metrics: {'accuracy': 0.7110501029512697, 'recall': 0.7110567991300014, 'precision': 0.7107091022185361, 'kappa': 0.42162204975357054, 'auc': 0.7848962345371335}
Average accuracy per test subject:
  v28p: 0.0189
  v274: 1.0000
  v1p: 0.3478
  v231: 0.8289
  v22p: 0.0435
  v29p: 0.7312
  v206: 0.9610
  v238: 1.0000
  v31p: 1.0000
  v35p: 0.9828
  v177: 1.0000
  v200: 1.0000
  v112: 0.8361
  v113: 0.8983
  v48p: 1.0000
  v140: 0.0000
  v131: 0.7031
  v125: 0.6441
  v55p: 0.6481
  v143: 0.1695
  v43p: 1.0000
  v305: 1.0000
  v134: 0.7843
  v114: 1.0000

------------------------------------------------------------
Fold 2/5  |  Test subjects: ['v18p', 'v39p', 'v234', 'v32p', 'v190', 'v6p', 'v254', 'v204', 'v24p', 'v183', 'v246', 'v219', 'v298', 'v41p', 'v47p', 'v308', 'v52p', 'v300', 'v59p', 'v299', 'v302', 'v51p', 'v109', 'v127']
------------------------------------------------------------
53/53 ━━━━━━━━━━━━━━━━━━━━ 5s 61ms/step

Fold 2 Metri

In [10]:
for i in range(5):
    result = results[i]
    accs = []
    for r in result:
        accs.append(r['accuracy'])
    
    print(i, '->', np.mean(accs))

0 -> 0.7375780626803354
1 -> 0.7375780626803354
2 -> 0.7375780626803354
3 -> 0.7375780626803354
4 -> 0.7375780626803354


In [11]:
import pickle

with open(f'results_L24SO_{model_name}.pkl', 'wb') as f:
    pickle.dump(results, f)